# Bangladesh: 0206_flood_histogram

Static adaptation only; Earth Engine/data/asset execution not verified.

Original scientific assumptions and date windows retained; use the national workflow for QA improvements.

Review scale, missing data, units, and any training/calibration requirements before interpreting results.

May contain tutorial-specific assets or imported training features; inspect dependencies before execution.

In [ ]:
# Bangladesh adaptation: 0206_flood_histogram
# Original author: Amirhossein Ahrari; see original attribution below and originals/.
from bdgee.core import initialize, boundaries
from bdgee.notebook import export_outputs, upload_local
initialize()  # Run `earthengine authenticate` first; project is configured in bdgee.
BD_COUNTRY, BD_DISTRICTS = boundaries()
BD_ROI = BD_COUNTRY.geometry()
BD_BBOX = [88.0, 20.5, 92.7, 26.7]  # search only, not analysis boundary
roi = geometry = border = iran = tehran = province = loc = point = BD_ROI
# Run from the repository root; all local outputs are isolated by source.
from pathlib import Path
import os
_REPO_ROOT = Path.cwd()
_OUTPUT_DIR = Path(os.environ.get('BDGEE_OUTPUT_DIR', _REPO_ROOT / 'outputs' / 'flood' / '0206_flood_histogram'))
_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(_OUTPUT_DIR)


In [ ]:
"""00006_flood_histogram.ipynb

Tutorial Code by Amirhossein Ahrari
YouTube: https://www.youtube.com/@amirhosseinahrarigee
Tutorial Video: Google Earth Engine Tutorial-195: Pre-Flood and Post-Flood Histogram Analysis
This code is part of a tutorial series on Earth Engine programming techniques
presented on the Amirhossein Ahrari YouTube channel. You are free to use and modify
this code for academic and non-academic purposes. Don't forget to subscribe to
the Amirhossein Ahrari channel and follow the videos to support the instructor


"""
import ee
None
None
'# **Study Area**'
import geemap
map = geemap.Map(basemap='SATELLITE')
map
roi = BD_ROI
roi
'# **Landsat Image Collection**'

def ndwi(img):
    qa_band = img.select('QA_PIXEL')
    bit1 = qa_band.bitwiseAnd(1 << 1).neq(0)
    bit2 = qa_band.bitwiseAnd(1 << 2).neq(0)
    bit3 = qa_band.bitwiseAnd(1 << 3).neq(0)
    bit4 = qa_band.bitwiseAnd(1 << 4).neq(0)
    mask = bit1.Or(bit2).Or(bit3).Or(bit4)
    sr = img.select('SR.*').multiply(2.75e-05).add(-0.2)
    index = sr.normalizedDifference(['SR_B3', 'SR_B5']).rename('ndwi')
    return index.updateMask(mask.Not()).copyProperties(img, ['system:time_start'])
landsat = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterDate('2018', '2020').filterBounds(roi)
landsat_ndwi = landsat.map(ndwi)
landsat_ndwi
'# **Image Collection to Xarray**'
from shapely.geometry import shape
roi_shape = shape(roi.getInfo())
roi_shape
from xee import helpers
landsat.first().select('SR_B4').projection()
grid_params = helpers.fit_geometry(geometry=roi_shape, grid_crs='EPSG:32639', grid_scale=(100, -100))
import xarray as xr
ds = xr.open_dataset(landsat_ndwi, engine='ee', **grid_params)
ds = ds.sortby('time') * 1
'# **Flood Analysis**'
baseline = ds.median(dim='time')
baseline.ndwi.plot(x='x', y='y', robust=True)
import matplotlib.pyplot as plt
baseline.ndwi.values.flatten()
plt.hist(baseline.ndwi.values.flatten(), bins=200)
plt.show()
flood = ds.sel(time='2019-04').max(dim='time')
flood.ndwi.plot(x='x', y='y', robust=True)
plt.hist(flood.ndwi.values.flatten(), bins=200)
plt.show()
'# **Visualization**'
fig, ax = plt.subplots(1, 3, figsize=(13, 4))
baseline.ndwi.plot(x='x', y='y', robust=True, ax=ax[0], cbar_kwargs={'orientation': 'horizontal', 'pad': 0.05, 'label': 'pre-flood'})
ax[0].tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
ax[0].set_ylabel('')
ax[0].set_xlabel('')
flood.ndwi.plot(x='x', y='y', robust=True, ax=ax[2], cbar_kwargs={'orientation': 'horizontal', 'pad': 0.05, 'label': 'post-flood'})
ax[2].tick_params(left=False, labelleft=False, bottom=False, labelbottom=False)
ax[2].set_ylabel('')
ax[2].set_xlabel('')
ax[1].hist(flood.ndwi.values.flatten(), bins=200, color='tab:blue', label='post-flood', alpha=0.5)
ax[1].hist(baseline.ndwi.values.flatten(), bins=200, color='tab:red', label='pre-flood', alpha=0.5)
ax[1].legend()
plt.savefig('flood_hist.png', dpi=360, bbox_inches='tight')
plt.show()


In [ ]:

# Review these task plans; task creation does not start exports.
os.chdir(_REPO_ROOT)
bd_tasks = export_outputs(globals(), 'flood', '0206_flood_histogram', scale=1000)
# Set an appropriate native scale above before creating plans.
# for task in bd_tasks: task.start()
# upload_local(_OUTPUT_DIR, 'flood')  # explicit upload; requires Drive OAuth scope
